# Paper Figures and Tables

This notebook runs entirely from this repository. Install the root `requirements.txt`, start a kernel from the repository root or any subdirectory, and run all cells in order. No model API calls or credentials are required.

- Summary inputs: `data/plots_and_tables/*.csv` and `appendix_tables/`.
- Raw EC inputs: `data/results/EC/`; stance labels: `code/EC/inputs/ai_regulation_minimal_personas_100.json`.
- Plotting code: `code/plots/`.
- Outputs: PNG/PDF figures in `data/plots_and_tables/figs/`; refreshed directional tables and EC design agreement/trajectory CSVs in `data/plots_and_tables/`.

The main table and appendix tables use bundled statistical exports; EC figures recompute paired metrics and bootstrap intervals from the included runs.

In [ ]:
from pathlib import Path


def find_repo_root():
    for directory in (Path.cwd(), *Path.cwd().parents):
        for candidate in (directory, directory / "ICLR_github"):
            if (candidate / "data" / "plots_and_tables" / "paper_visualizations.ipynb").is_file():
                return candidate
    raise FileNotFoundError("Run this notebook from the repository root or one of its subdirectories.")


ROOT = find_repo_root()
DATA_DIR = ROOT / "data" / "plots_and_tables"
RESULTS_DIR = ROOT / "data" / "results"
FIG_DIR = DATA_DIR / "figs"
FIG_DIR.mkdir(exist_ok=True)
print(f"Repository: {ROOT}")

# Table: Cross-experiment sensitivity summary

PD cells select the largest absolute prespecified pairwise contrast separately for each metric, then report that contrast's Holm-corrected significance. All EC design cells use the latest 30 seed-paired runs per condition, with 15 rounds per run, and report higher minus lower: activation 0.5 minus 0.3, memory window 4 minus 2, and recommendation feed size 10 minus 5. Design inference uses two-sided paired t-tests, with Holm correction across the three design perturbations within each model-outcome family. Prompt-representation cells report Narrative Prose minus Bullet Original using two-sided seed-paired Wilcoxon signed-rank tests, with Holm correction across the eight model-outcome tests. DeepSeek uses V3.2 for PD and V4 Flash for all EC perturbations. Payoff is measured in original payoff units, cooperation in percentage points, and EC outcomes in their native network-metric units. Unmarked effects are not significant after Holm correction.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

OUTPUT_DIR = DATA_DIR
table_path = DATA_DIR / "sensitivity_summary_values.csv"
table_values = pd.read_csv(table_path)
prompt_results = table_values.loc[table_values["perturbation"].eq("Prompt representation")].copy()

designs = ["Activation probability", "Memory window", "Feed size"]
design_results = table_values.loc[table_values["perturbation"].isin(designs)].copy()
feed_results = design_results.loc[design_results["perturbation"].eq("Feed size")].copy()
memory_results = design_results.loc[design_results["perturbation"].eq("Memory window")].copy()

deepseek_display = "DeepSeek (V3.2 PD; V4 Flash EC)"
display_values = table_values.copy()
display_values["model"] = display_values["model"].replace({
    "DeepSeek V3.2": deepseek_display,
    "DeepSeek V3": deepseek_display,
    "DeepSeek V4 Flash": deepseek_display,
})
models = ["GPT-5.2", "Claude Haiku 4.5", "Gemini 2.5 Flash", deepseek_display]
rows = [
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Persona", "Average payoff"),
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Persona", "Cooperation rate (pp)"),
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Game framing", "Average payoff"),
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Game framing", "Cooperation rate (pp)"),
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Memory", "Average payoff"),
    ("Repeated Prisoner's Dilemma (Two-Agent)", "Memory", "Cooperation rate (pp)"),
    ("Social-Media Echo Chamber", "Activation probability", "Stance assortativity"),
    ("Social-Media Echo Chamber", "Activation probability", "Same-group edge ratio"),
    ("Social-Media Echo Chamber", "Memory window", "Stance assortativity"),
    ("Social-Media Echo Chamber", "Memory window", "Same-group edge ratio"),
    ("Social-Media Echo Chamber", "Feed size", "Stance assortativity"),
    ("Social-Media Echo Chamber", "Feed size", "Same-group edge ratio"),
    ("Social-Media Echo Chamber", "Prompt representation", "Stance assortativity"),
    ("Social-Media Echo Chamber", "Prompt representation", "Same-group edge ratio"),
]
names = {
    "Persona": "Persona format", "Game framing": "Game framing",
    "Memory": "Memory representation", "Activation probability": "Activation probability",
    "Memory window": "Memory window", "Feed size": "Recommendation feed size",
    "Prompt representation": "Persona representation",
}


def format_value(row):
    effect = row["signed_effect"] * 100 if row["metric"] == "Cooperation rate (pp)" else row["signed_effect"]
    decimals = 1 if row["metric"] == "Cooperation rate (pp)" else 3 if row["experiment"] == "Social-Media Echo Chamber" else 2
    stars = row["significance"] if pd.notna(row["significance"]) else ""
    return f"{'-' if effect < 0 else '+'}{abs(effect):.{decimals}f}{stars}"


display_rows = []
for experiment, perturbation, metric in rows:
    row = {"Experiment": experiment, "Perturbation": names[perturbation], "Metric": metric}
    for model in models:
        match = display_values[
            (display_values["experiment"] == experiment)
            & (display_values["perturbation"] == perturbation)
            & (display_values["metric"] == metric)
            & (display_values["model"] == model)
        ]
        assert len(match) == 1
        row[model] = format_value(match.iloc[0])
    display_rows.append(row)
table_display = pd.DataFrame(display_rows).set_index(["Experiment", "Perturbation", "Metric"])

assert table_display.shape == (14, 4) and len(table_values) == 56
assert len(design_results) == 24 and design_results[["n_0", "n_1"]].eq(30).all().all()
assert len(feed_results) == 8 and len(memory_results) == 8
assert len(prompt_results) == 8 and prompt_results[["n_0", "n_1"]].eq(30).all().all()
assert not table_values.duplicated(["experiment", "perturbation", "metric", "model"]).any()
assert design_results[["ci_low", "ci_high", "holm_p"]].notna().all().all()
assert prompt_results["holm_p"].notna().all()


print("Table 3. Outcome sensitivity to representation- and design-level perturbations across models")
display(table_display)
print("PD payoff: original payoff units; PD cooperation: percentage points; EC: native network-metric units.")
print("PD cells use the largest absolute prespecified contrast. EC design cells use higher minus lower.")
print("All EC design contrasts use N=30 paired seeds per condition, with 15 rounds per run.")
print("EC design inference: paired t-tests; Holm correction across activation, memory, and feed within each model-outcome family.")
print("Prompt representation uses Narrative Prose minus Bullet Original (N=30; paired Wilcoxon tests; Holm across eight tests).")
print("DeepSeek uses V3.2 for PD and V4 Flash for all EC perturbations.")
print("Activation is 0.5-0.3 for all models; memory is 4-2; feed is 10-5.")
print("Stars use Holm-corrected p-values: *** p<.001, ** p<.01, * p<.05; unmarked values are non-significant.")
print("Validation passed: 14 rows, 4 model columns, 56 unique source values, and N=30 for all EC results.")
display(design_results[[
    "model", "perturbation", "metric", "mean_0", "mean_1", "signed_effect",
    "ci_low", "ci_high", "raw_p", "holm_p", "effect_size", "significance", "n_0", "n_1",
]])
display(prompt_results[[
    "model", "metric", "mean_0", "mean_1", "signed_effect", "ci_low", "ci_high",
    "raw_p", "holm_p", "effect_size", "significance",
]])

# Extended Table: All pairwise sensitivity contrasts

Appendix companion to Table 3. Rows identify each prespecified contrast and outcome; model columns report $\Delta$ [95% CI], where $\Delta=\mathrm{Condition\ 1}-\mathrm{Condition\ 0}$. Stars denote Holm-adjusted significance. Cooperation-rate effects are in percentage points; payoff and EC outcomes retain their native units. DeepSeek uses V3.2 for PD and V4 Flash for all EC perturbations. All EC contrasts use 30 paired seeds per condition and 15 rounds per run. Activation contrasts are 0.3 to 0.5, memory-window contrasts are 2 to 4, and recommendation-feed contrasts are 5 to 10 for all four models.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

extended_table_dir = DATA_DIR
pd_pairwise = pd.read_csv(extended_table_dir / "pd_all_pairwise_results.csv")
ec_pairwise = pd.read_csv(extended_table_dir / "sensitivity_summary_values.csv").query(
    "experiment == 'Social-Media Echo Chamber'"
)
extended_pairwise = pd.concat([pd_pairwise, ec_pairwise], ignore_index=True)

# Every reported effect is directional: Condition 1 minus Condition 0.
assert np.allclose(
    extended_pairwise["signed_effect"],
    extended_pairwise["mean_1"] - extended_pairwise["mean_0"],
)
assert len(pd_pairwise) == 96 and len(ec_pairwise) == 32 and len(extended_pairwise) == 128
assert ec_pairwise[["n_0", "n_1"]].eq(30).all().all()
assert not extended_pairwise.duplicated(
    ["experiment", "perturbation", "metric", "model", "condition_0", "condition_1"]
).any()
assert extended_pairwise[["ci_low", "ci_high", "holm_p"]].notna().all().all()

# Cooperation rates are stored as proportions; report them as percentage points.
cooperation_mask = extended_pairwise["metric"].eq("Cooperation rate (pp)")
scale_columns = ["mean_0", "mean_1", "signed_effect", "ci_low", "ci_high"]
extended_pairwise.loc[cooperation_mask, scale_columns] *= 100

experiment_order = {
    "Repeated Prisoner's Dilemma (Two-Agent)": 0,
    "Social-Media Echo Chamber": 1,
}
perturbation_order = {
    "Persona": 0,
    "Game framing": 1,
    "Memory": 2,
    "Activation probability": 3,
    "Memory window": 4,
    "Feed size": 5,
    "Prompt representation": 6,
}
model_order = {
    "GPT-5.2": 0,
    "Claude Haiku 4.5": 1,
    "Gemini 2.5 Flash": 2,
    "DeepSeek V3": 3,
    "DeepSeek V3.2": 3,
    "DeepSeek V4 Flash": 3,
}
deepseek_display = "DeepSeek (V3.2 PD; V4 Flash EC)"
model_display = {
    "GPT-5.2": "GPT-5.2",
    "Claude Haiku 4.5": "Claude Haiku 4.5",
    "Gemini 2.5 Flash": "Gemini 2.5 Flash",
    "DeepSeek V3": deepseek_display,
    "DeepSeek V3.2": deepseek_display,
    "DeepSeek V4 Flash": deepseek_display,
}
extended_pairwise["_experiment_order"] = extended_pairwise["experiment"].map(experiment_order)
extended_pairwise["_perturbation_order"] = extended_pairwise["perturbation"].map(perturbation_order)
extended_pairwise["_model_order"] = extended_pairwise["model"].map(model_order)
extended_pairwise = extended_pairwise.sort_values(
    ["_experiment_order", "_perturbation_order", "metric", "condition_0", "condition_1", "_model_order"]
).reset_index(drop=True)

# Keep the full numeric long table as the machine-readable appendix source.
extended_table = extended_pairwise.rename(
    columns={
        "experiment": "Experiment",
        "perturbation": "Perturbation",
        "metric": "Outcome",
        "model": "Model",
        "condition_0": "Condition 0",
        "condition_1": "Condition 1",
        "mean_0": "Mean 0",
        "mean_1": "Mean 1",
        "signed_effect": "Delta (Condition 1 - Condition 0)",
        "ci_low": "95% CI lower",
        "ci_high": "95% CI upper",
        "effect_size": "Paired effect size",
        "raw_p": "Raw p",
        "holm_p": "Holm-adjusted p",
        "significance": "Significance",
        "n_0": "N 0",
        "n_1": "N 1",
    }
)[
    [
        "Experiment", "Perturbation", "Outcome", "Model", "Condition 0", "Condition 1",
        "Mean 0", "Mean 1", "Delta (Condition 1 - Condition 0)",
        "95% CI lower", "95% CI upper", "Paired effect size",
        "Raw p", "Holm-adjusted p", "Significance", "N 0", "N 1",
    ]
]
extended_csv_path = extended_table_dir / "all_directional_pairwise_results.csv"
extended_table.to_csv(extended_csv_path, index=False)

# Compress models into columns while retaining every directional contrast.
def format_directional_result(row):
    stars = row["significance"] if pd.notna(row["significance"]) else ""
    return (
        f"{row['signed_effect']:+.3f} "
        f"[{row['ci_low']:+.3f}, {row['ci_high']:+.3f}]{stars}"
    )


extended_pairwise["Contrast"] = (
    extended_pairwise["condition_0"].astype(str)
    + " -> "
    + extended_pairwise["condition_1"].astype(str)
)
extended_pairwise["Model column"] = extended_pairwise["model"].map(model_display)
extended_pairwise["Directional delta [95% CI]"] = extended_pairwise.apply(
    format_directional_result, axis=1
)

compact_table = extended_pairwise.pivot(
    index=["experiment", "perturbation", "metric", "Contrast"],
    columns="Model column",
    values="Directional delta [95% CI]",
).reset_index()
compact_table.columns.name = None
compact_table["_experiment_order"] = compact_table["experiment"].map(experiment_order)
compact_table["_perturbation_order"] = compact_table["perturbation"].map(perturbation_order)
compact_table = compact_table.sort_values(
    ["_experiment_order", "_perturbation_order", "metric", "Contrast"]
).drop(columns=["_experiment_order", "_perturbation_order"])
compact_table = compact_table.rename(
    columns={
        "experiment": "Experiment",
        "perturbation": "Perturbation",
        "metric": "Outcome",
    }
)[
    [
        "Experiment", "Perturbation", "Outcome", "Contrast",
        "GPT-5.2", "Claude Haiku 4.5", "Gemini 2.5 Flash", deepseek_display,
    ]
].reset_index(drop=True)

assert len(compact_table) == 32
assert compact_table[["GPT-5.2", "Claude Haiku 4.5", "Gemini 2.5 Flash", deepseek_display]].notna().sum().to_dict() == {
    "GPT-5.2": 32,
    "Claude Haiku 4.5": 32,
    "Gemini 2.5 Flash": 32,
    deepseek_display: 32,
}
compact_csv_path = extended_table_dir / "compact_directional_pairwise_results.csv"
compact_table.to_csv(compact_csv_path, index=False)

print("Extended Table. All directional pairwise sensitivity contrasts")
print("Each model cell reports Delta [95% CI], where Delta = Condition 1 - Condition 0.")
print("Cooperation outcomes are in percentage points; stars use Holm-adjusted p-values.")
print("DeepSeek uses V3.2 for PD and V4 Flash for all EC perturbations.")
print("All EC contrasts use N=30 paired seeds per condition, with 15 rounds per run.")
print(f"Saved {len(extended_table)}-row numeric source: {extended_csv_path}")
print(f"Saved {len(compact_table)}-row compact table: {compact_csv_path}")
display(compact_table.fillna("—"))

# Figure: Interaction changes representation sensitivity

**a**, Representation sensitivity under fixed-opponent and reciprocal LLM–LLM interaction regimes, denoted $S^{\mathrm{fixed}}$ and $S^{\mathrm{LLM}}$. **b**, Signed interaction effect gap, $A=S^{\mathrm{LLM}}-S^{\mathrm{fixed}}$, with paired-seed bootstrap 95% confidence intervals; $A>0$ indicates amplification and $A<0$ attenuation. **c**, Round-wise interaction effect $A_t$; shaded bands are pointwise paired-bootstrap 95% confidence intervals. All panels show the Plain → Descriptive contrast.

In [ ]:
from pathlib import Path
import importlib
import sys

amplification_dir = DATA_DIR
figure_output_dir = FIG_DIR
plot_module_dir = ROOT / "code" / "plots"
if str(plot_module_dir) not in sys.path:
    sys.path.insert(0, str(plot_module_dir))

import plot_interaction_modulation as interaction_plot
importlib.reload(interaction_plot)

figure_paths = interaction_plot.plot_interaction_modulation_panels(
    amplification_dir,
    figure_output_dir,
    dimension="persona",
    contrast="Plain -> Descriptive",
    effect_axis_label="Representation sensitivity (pp)",
    output_stem="pd_persona_amplification",
    magnitude_xlim=(-3, 86),
    magnitude_ticks=[0, 20, 40, 60, 80],
    gap_xlim=(-18, 68),
    gap_ticks=[-10, 0, 20, 40, 60],
    trajectory_ylim=(-50, 85),
    trajectory_ticks=[-40, -20, 0, 20, 40, 60, 80],
)
for png_path, pdf_path in figure_paths:
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

# Figures: EC persona-representation sensitivity

Three independently saved 4.5 × 4.5 inch panels for GPT-5.2, Claude Haiku 4.5, Gemini 2.5 Flash, and DeepSeek V4 Flash: **a**, paired-seed action and target agreement, with each model's overall Narrative - Bullet cross-group targeting difference; **b**, round-wise targeting sensitivity, $D_{\mathrm{target},t}=q_t^{\mathrm{Narrative}}-q_t^{\mathrm{Bullet}}$; and **c**, collective divergence in cumulative weighted stance assortativity, $D_{r,t}=r_t^{\mathrm{Narrative}}-r_t^{\mathrm{Bullet}}$. Bands are pointwise 95% paired-percentile-bootstrap confidence intervals across 30 seeds per model.

In [ ]:
from pathlib import Path
import importlib
import sys

ec_figure_dir = RESULTS_DIR / "EC"
figure_output_dir = FIG_DIR
plot_module_dir = ROOT / "code" / "plots"
if str(plot_module_dir) not in sys.path:
    sys.path.insert(0, str(plot_module_dir))

import plot_main_representation_figure as ec_representation_plot
importlib.reload(ec_representation_plot)

ec_figure_paths = ec_representation_plot.plot_ec_representation_sensitivity(
    ec_figure_dir,
    figure_output_dir,
)
for png_path, pdf_path in ec_figure_paths:
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

# Figures: EC design perturbation sensitivity

Use the same three-panel metrics and plotting method as the EC persona-representation figures for each design perturbation: **Activation probability**, 0.5 minus 0.3; **Memory window**, 4 minus 2; and **Recommendation feed size**, 10 minus 5. All comparisons use 30 paired seeds per model and 15 rounds per run, with GPT-5.2, Claude Haiku 4.5, Gemini 2.5 Flash, and DeepSeek V4 Flash.

For each perturbation, **a** shows action-type and interaction-target agreement over matched seed-round-agent decisions; **b** shows the round-wise cross-group targeting difference, $\Delta_{\mathrm{target},t}=q_t^{\mathrm{higher}}-q_t^{\mathrm{lower}}$, in percentage points; and **c** shows collective divergence in cumulative weighted stance assortativity, $\Delta_{r,t}=r_t^{\mathrm{higher}}-r_t^{\mathrm{lower}}$. Agreement percentages, model colors and markers, axis labels, 4.5 x 4.5 inch panel dimensions, and paired-percentile-bootstrap inference match the persona-representation figures. Intervals use 10,000 seed resamples. Trajectory bands are pointwise 95% CIs, not simultaneous or multiplicity-adjusted. Round-1 stance assortativity remains undefined because there are no eligible interaction edges. Activation agreement includes only agent-round decisions present in both conditions.

Nine independent panels are saved as PNG/PDF pairs. Cross-group targeting comes from each run's round-summary interaction ratio; it is not the cumulative same-group edge ratio reported in Table 3. The table itself is unchanged.

In [ ]:
from pathlib import Path
import importlib
import sys

design_figure_dir = RESULTS_DIR / "EC"
design_output_dir = FIG_DIR
plot_module_dir = ROOT / "code" / "plots"
if str(plot_module_dir) not in sys.path:
    sys.path.insert(0, str(plot_module_dir))

import plot_main_representation_figure as ec_design_plot
importlib.reload(ec_design_plot)

design_figure_paths = ec_design_plot.plot_ec_design_perturbation_sensitivity(
    design_figure_dir,
    design_output_dir,
)
for png_path, pdf_path in design_figure_paths:
    print(f"Saved: {png_path}")
    print(f"Saved: {pdf_path}")

assert len(design_figure_paths) == 9

# Extended Experimental Results

## Complete Outcome-Sensitivity Results

Table A reports every prespecified two-agent Prisoner's Dilemma contrast underlying the main-text sensitivity summary. The dagger marks the maximum absolute contrast selected for the corresponding model, dimension, and outcome.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

appendix_dir = DATA_DIR / "appendix_tables"

appendix_tables = {}
appendix_sections = [
    ("A", "Complete Outcome-Sensitivity Results", "pd_complete_representation_sensitivity"),
    ("B", "Interaction Amplification in the Prisoner's Dilemma", "pd_complete_amplification"),
    ("C", "Representation Sensitivity in the Echo-Chamber Simulation", "ec_representation_sensitivity"),
    ("D", "Interaction-Pathway Diagnostics", "ec_exposure_diagnostic"),
]
for label, heading, table_name in appendix_sections:
    if label != "A":
        display(Markdown(f"## {heading}"))
    csv_path = appendix_dir / f"{table_name}.csv"
    tex_path = appendix_dir / f"{table_name}.tex"
    table = pd.read_csv(csv_path)
    appendix_tables[label] = table
    print(f"Table {label}: {len(table)} rows")
    print(f"CSV: {csv_path}")
    print(f"LaTeX: {tex_path}")
    display(table)